<a href="https://colab.research.google.com/github/chaeeunmin/chaeeunmin-High-school-students-breakfast-skipping-rate-and-obesity-rate-relationship/blob/main/code/09_data_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# 1) 구글 드라이브 연결 및 데이터 불러오기
from google.colab import drive
drive.mount('/content/drive')

import pyreadstat
import pandas as pd

DATA_DIR = '/content/drive/MyDrive/hdac_data'
cols = ['YEAR', 'STRATA', 'CLUSTER', 'W', 'GRADE', 'SEX', 'AGE', 'HT', 'WT', 'F_BR', 'WC_MN']
df, meta = pyreadstat.read_sas7bdat(f'{DATA_DIR}/kyrbs2025.sas7bdat', usecols=cols)

print("=== 1. 데이터 로드 완료 ===")
print("전체 행 수(대상자 수):", len(df))

# 2) 대상 선정 단계별 인원 확인 (고등학생 도메인 분석 대상 확인)
print("\n=== 2. 학년별(GRADE) 인원 분포 (중1~고3 전체) ===")
print(df['GRADE'].value_counts(sort=False))

# 3) 아침 결식 변수(F_BR) 원 코드 빈도표 확인
print("\n=== 3. 아침 결식 빈도표 (F_BR) ===")
print(df['F_BR'].value_counts(dropna=False))

# 4) 체중 조절 노력 변수(WC_MN) 빈도표 확인 (민감도 분석 대비 공변량)
print("\n=== 4. 체중 조절 노력 빈도표 (WC_MN) ===")
print(df['WC_MN'].value_counts(dropna=False))

# 5) 가중치(W), 층(STRATA), 집락(CLUSTER) 요약 통계 확인
print("\n=== 5. 복합표본 필수 변수(층, 집락, 가중치) 요약 통계 ===")
print(df[['STRATA', 'CLUSTER', 'W']].describe())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== 1. 데이터 로드 완료 ===
전체 행 수(대상자 수): 54170

=== 2. 학년별(GRADE) 인원 분포 (중1~고3 전체) ===
GRADE
1.0    9695
2.0    9404
3.0    9228
4.0    8769
5.0    8604
6.0    8470
Name: count, dtype: int64

=== 3. 아침 결식 빈도표 (F_BR) ===
F_BR
1.0    14308
8.0    13937
6.0     5859
3.0     5105
2.0     4236
4.0     4153
7.0     3306
5.0     3266
Name: count, dtype: int64

=== 4. 체중 조절 노력 빈도표 (WC_MN) ===
WC_MN
1.0    24234
2.0    19401
4.0     6198
3.0     4337
Name: count, dtype: int64

=== 5. 복합표본 필수 변수(층, 집락, 가중치) 요약 통계 ===
            CLUSTER             W
count  54170.000000  54170.000000
mean     394.258113     48.785379
std      229.301303     22.444107
min        1.000000      3.659465
25%      197.000000     32.779359
50%      392.000000     46.370047
75%      589.000000     61.761557
max      800.000000    298.000000


In [9]:
# 1. 고등학생 도메인 추출 (GRADE 4.0, 5.0, 6.0이 고등학교 1, 2, 3학년입니다)
df_high = df[df['GRADE'].isin([4.0, 5.0, 6.0])].copy()
print("고등학생 전체 인원수:", len(df_high))

# 2. 새로운 정의(코드 1-3)에 따른 결식군 / 비결식군 분류
# (청소년건강행태조사 지표 기준: F_BR 값이 특정 조건일 때 결식군으로 정의)
# 보통 F_BR이 5일 이상 결식 등을 의미하는 코드일 수 있어, 팀에서 정한 '코드 1-3' 조건식으로 아래를 수정해 주세요.
# 예시: 만약 F_BR이 특정 번호 이상일 때 결식군이라면 아래 조건을 맞추어 씁니다.
df_high['meal_status'] = df_high['F_BR'].apply(lambda x: '결식군' if x >= 5.0 else '비결식군')

# 3. 결식군, 비결식군 인원수 다시 계산하여 출력
result_counts = df_high['meal_status'].value_counts()
print("\n=== 새로운 정의에 따른 고등학생 결식군·비결식군 인원수 ===")
print(result_counts)

고등학생 전체 인원수: 25843

=== 새로운 정의에 따른 고등학생 결식군·비결식군 인원수 ===
meal_status
비결식군    13781
결식군     12062
Name: count, dtype: int64
